# Tutorial 03: Clasificación de Ritmos EEG y Destilación de Conocimiento (Knowledge Distillation)
### TinyML con PyTorch para Raspberry Pi Pico 2 / Pico 2 W (RP2350 - Cortex-M33)

En este tutorial aprenderás:
1. **Generación de ritmos cerebrales sintéticos (EEG)** usando `neurokit2` y modelos oscilatorios: Theta (4-8 Hz), Alpha (8-12 Hz) y Beta (14-28 Hz).
2. **Teoría de Knowledge Distillation (Hinton et al.)**:
   - **Teacher Model**: Red profunda de alta capacidad (~300,000 parámetros) que aprende representaciones complejas y "dark knowledge" (probabilidades suaves entre clases).
   - **Student Model**: Red ultra-compacta (<3,000 parámetros) diseñada para ejecutarse en microcontroladores con pocos kilobytes de RAM.
   - **Función de Pérdida con Temperatura $T$**:
     $$\mathcal{L}_{KD} = \alpha T^2 D_{\text{KL}}\left(\sigma(z_s / T) \parallel \sigma(z_t / T)\right) + (1-\alpha) \mathcal{L}_{CE}(z_s, y)$$
3. **Conversión cruzada PyTorch -> TFLite INT8**.
4. **Exportación a encabezados C (`model_data.h`) e inferencia en Arduino IDE sobre Raspberry Pi Pico 2**.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path("../../").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from utils.signal_generator import generate_eeg_dataset
from utils.exporter import tflite_to_c_header, save_test_samples_header

print(f"PyTorch Version: {torch.__version__}")

## 1. Generación de Señales EEG Sintéticas (Bandas de Frecuencia)

In [ ]:
SAMPLE_LEN = 256
SAMPLING_RATE = 128
CLASS_NAMES = ["Theta (4-8Hz)", "Alpha (8-12Hz)", "Beta (14-28Hz)"]

X, y = generate_eeg_dataset(num_samples_per_class=400, sample_len=SAMPLE_LEN, sampling_rate=SAMPLING_RATE)
X = np.expand_dims(X, axis=1) # (N, 1, 256)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5), sharey=True)
t_axis = np.arange(SAMPLE_LEN) / SAMPLING_RATE
for c in range(3):
    idx = np.where(y_train == c)[0][0]
    axes[c].plot(t_axis, X_train[idx, 0, :], color=["purple", "teal", "darkorange"][c])
    axes[c].set_title(f"{CLASS_NAMES[c]}", fontweight='bold')
    axes[c].set_xlabel("Tiempo (s)")
    axes[c].grid(True, alpha=0.5)
axes[0].set_ylabel("Voltaje Normalizado (uV)")
plt.tight_layout()
plt.show()

## 2. Definición del Modelo Profesor (Teacher) y Modelo Alumno (Student)

In [ ]:
class DeepTeacherCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 64, 7, stride=2, padding=3)
        self.bn1 = nn.BatchNorm1d(64)
        self.conv2 = nn.Conv1d(64, 128, 5, stride=2, padding=2)
        self.bn2 = nn.BatchNorm1d(128)
        self.conv3 = nn.Conv1d(128, 256, 3, stride=2, padding=1)
        self.bn3 = nn.BatchNorm1d(256)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc1 = nn.Linear(256, 128)
        self.fc2 = nn.Linear(128, 3)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.pool(x).squeeze(-1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

class TinyStudentCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 8, 7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(8)
        self.conv2 = nn.Conv1d(8, 16, 5, stride=2, padding=2, bias=False)
        self.bn2 = nn.BatchNorm1d(16)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(16, 3)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool(x).squeeze(-1)
        return self.fc(x)

teacher = DeepTeacherCNN()
student = TinyStudentCNN()
print(f"Parámetros Teacher: {sum(p.numel() for p in teacher.parameters()):,}")
print(f"Parámetros Student: {sum(p.numel() for p in student.parameters()):,}")

## 3. Entrenamiento con Pérdida de Destilación (Knowledge Distillation)

In [ ]:
train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
test_ds = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long))
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

# 1. Entrenar Teacher
opt_t = torch.optim.Adam(teacher.parameters(), lr=0.002)
for ep in range(10):
    for x_b, y_b in train_loader:
        opt_t.zero_grad()
        F.cross_entropy(teacher(x_b), y_b).backward()
        opt_t.step()

# 2. Entrenar Student con Destilación
opt_s = torch.optim.Adam(student.parameters(), lr=0.003)
T = 3.0
alpha = 0.6
for ep in range(12):
    for x_b, y_b in train_loader:
        opt_s.zero_grad()
        with torch.no_grad():
            t_logits = teacher(x_b)
        s_logits = student(x_b)
        loss_ce = F.cross_entropy(s_logits, y_b)
        loss_kd = F.kl_div(F.log_softmax(s_logits/T, dim=1), F.softmax(t_logits/T, dim=1), reduction="batchmean") * (T**2)
        (alpha * loss_kd + (1 - alpha) * loss_ce).backward()
        opt_s.step()

student.eval()
correct = sum((student(x_b).argmax(dim=1) == y_b).sum().item() for x_b, y_b in test_loader)
print(f"Exactitud del Alumno Destilado: {correct / len(test_ds) * 100:.2f}%")